Group XX | Members: [Student 1], [Student 2] | Question: Q1 | Stage: 6: finalise

How this works: package final outputs, write data card/memo/checklists, and expose a reusable prepare_new_data interface.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pathlib import Path
from datetime import date
import pandas as pd
import joblib
from src.config import DOCS_DIR, PROCESSED_DIR, INTERIM_DIR, TARGET_COL
from src.cleaning import clean_q1, add_time_features
from src.features import engineer_features
from src.logging_utils import CleaningLog

# prepare_new_data helper
prep_py = ROOT / 'src/prepare_new_data.py'
prep_py.write_text('''import pandas as pd
from src.cleaning import clean_q1, add_time_features
from src.features import engineer_features
from src.logging_utils import CleaningLog

def prepare_new_data(df_raw: pd.DataFrame) -> pd.DataFrame:
    clog = CleaningLog()
    clean = clean_q1(df_raw.copy(), clog)
    clean = add_time_features(clean)
    clean = engineer_features(clean)
    return clean
''')

train = pd.read_csv(PROCESSED_DIR / 'train_clean_unscaled.csv')
val = pd.read_csv(PROCESSED_DIR / 'validation_clean_unscaled.csv')
test = pd.read_csv(PROCESSED_DIR / 'test_clean_unscaled.csv')

rows_in = len(pd.read_csv(INTERIM_DIR / 'maize_clean.csv')) if (INTERIM_DIR / 'maize_clean.csv').exists() else 'n/a'
rows_usable = len(train) + len(val) + len(test)

data_card = f'''# Data Card (Q1 Maize)

- Purpose: prepare a machine-learning-ready maize dataset for questions A-E.
- Source: MCSU merged district forms (or stand-in when source unavailable).
- Size (usable): {rows_usable} rows across train/validation/test.
- Target: `{TARGET_COL}`.
- Key cleaning: text normalization, unit conversion to hectares, date parsing, deduplication, impossible value handling, leakage control.
- Key features: fertilizer per hectare, planting timing, seed and irrigation indicators, climate interactions.
- Limitations: data-entry noise, potential district/cooperative reporting bias, stand-in usage if real file missing.
- Reproducibility: `python run_all.py`.
'''
(DOCS_DIR / 'data_card.md').write_text(data_card)

memo = f'''# Memo to Manager

**To:** Mr. Habimana, Head of Planning and Data  
**From:** Junior Analyst (Group XX)  
**Date:** {date.today().isoformat()}  
**Subject:** Q1 Maize dataset readiness for ML analysis

1. **What we received:** We processed the raw file into clean train/validation/test datasets totaling {rows_usable} usable records.
2. **Main problems found:** (i) duplicate and near-duplicate farm-season records, (ii) mixed units and numeric text formats, (iii) unrealistic/invalid values and missingness. These issues would bias estimates if not corrected.
3. **Data removed/changed:** Invalid targets were removed, inconsistent target values were recomputed from harvest and area when inputs were trustworthy, and leakage column `harvest_weight_kg` was removed from model features.
4. **New features for questions A-E:** Added fertilizer per hectare, planting time metrics, seed quality indicators, altitude/temperature/rainfall interactions, and low-yield risk label.
5. **Limits and risks:** Quality still depends on field-entry consistency and district reporting practices; risk analyses must avoid post-harvest leakage.
6. **Recommendation:** Dataset is ready for ML preparation tasks for A-E, with strongest support for A-D and leakage-safe setup for E. Next season, standardize digital forms and validation rules at capture time.
'''
(DOCS_DIR / 'memo_to_manager.md').write_text(memo)

screenshot = '''# Screenshot Checklist (take manually)

- Stage 1 (`00_environment_test.ipynb`): versions output cell.
- Stage 2 (`01_acquire.ipynb`): file detection + shape output, data dictionary preview.
- Stage 3 (`02_eda.ipynb`): data quality report table output and at least 5 figure outputs (include notebook title cell).
- Stage 4 (`03_clean.ipynb`): split shapes, overlap check, low-yield threshold, cleaning log sample.
- Stage 5 (`04_features.ipynb`): feature counts, MI table, RF importance table.
- Stage 6 (`05_finalise.ipynb`): confirmation of generated docs and helper function.

Suggested caption format: "Stage X output showing [what], confirming [decision]."
Suggested explanation: 1-3 sentences on why the output matters for data readiness and leakage prevention.
'''
(DOCS_DIR / 'screenshot_checklist.md').write_text(screenshot)

cheat = '''# Oral Defence Cheatsheet

- Why split before scaling/imputation? To avoid test leakage and preserve unbiased validation.
- Why train-only fitting? Model preprocessing must reflect only information available at training time.
- Why median imputation? Numeric fields were skewed and had outliers.
- Why one-hot vs ordinal? Ordinal used only for pest levels (Low<Medium<High); nominal categories one-hot encoded.
- Why remove harvest_weight from features? It is post-harvest and leaks outcome information.
- Why use fertilizer per hectare? Fertilizer totals are not comparable across different farm sizes.
- Why stratify by season? Preserves seasonal distribution across train/val/test.
- Why cap outliers? Limits data-entry error influence while retaining plausible extremes.
'''
(DOCS_DIR / 'oral_defence_cheatsheet.md').write_text(cheat)

# update root documentation outline with Q1 drafted sections
root_outline = ROOT / 'docs/documentation_report_outline.md'
root_outline.write_text('''# GroupXX_ML_Element1_Documentation Outline

1. Cover page
2. Q1 Stage 1: Environment and reproducibility setup
3. Q1 Stage 2: Requirements, acquisition, data dictionary, source log
4. Q1 Stage 3: EDA findings and data quality report
5. Q1 Stage 4: Cleaning/preprocessing pipeline, splits, and leakage controls
6. Q1 Stage 5: Feature engineering and selection evidence
7. Q1 Stage 6: Final datasets, data card, memo, verification
8. Q2 scaffold and scenario documentation (to implement later)
9. Appendices (cleaning log, feature justification, screenshot checklist, oral defence notes)
''')

print('Final docs generated.')


Final docs generated.
